# MangaTR: فحص الـ GPU

بيعمل 4 حاجات ويطلّع تقرير واحد تنسخه وتبعتهولي:

1. بيسجّل الجهاز: اسم الـ GPU والـ VRAM والرام وإصدارات onnxruntime والـ providers المتاحة.
2. بيشغّل اختبارات المشروع (من غير GPU) للتأكد إن الكود سليم على بايثون بتاع Colab.
3. بينزّل 3 موديلات (الكشف، AOT، LaMa للمانجا) ويتأكد من الـ hash.
4. بيشغّل كل موديل على الـ GPU ويقيس الزمن (وعلى CPU للمقارنة)، **وبيقول لو onnxruntime رجع للـ CPU بصمت**.

الأزمنة بمدخلات عشوائية بنفس شكل مدخلات الموديل: بتقيس الموديل والجهاز، مش جودة أي نتيجة.

**قبل ما تبدأ:** `Runtime` ← `Change runtime type` ← **T4 GPU**. وضيف سر باسم `GITHUB_TOKEN` من أيقونة المفتاح على الشمال (الخطوة الجاية).


## التوكن (مرة واحدة)

الريبو خاص، فـ Colab محتاج توكن للقراءة بس:

1. GitHub ← Settings ← Developer settings ← Personal access tokens ← **Fine-grained tokens** ← Generate new token.
2. Repository access: **Only select repositories** ← `MangaTR`.
3. Permissions ← Repository permissions ← **Contents: Read-only**.
4. انسخ التوكن وحطه في Colab: أيقونة المفتاح ← Add new secret ← الاسم `GITHUB_TOKEN` ← فعّل Notebook access.

**ما تلصقش التوكن في أي خلية أو في الشات.** (ولو الريبو بقى عام مش محتاج التوكن.)


In [ ]:
# 1) الـ GPU
import subprocess

out = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
print(out or "مفيش GPU: غيّر نوع الـ Runtime إلى T4 ثم شغّل الخلية تاني")

In [ ]:
# 2) تحميل الكود
import os
import subprocess

from google.colab import userdata

BRANCH = "phase1/core-logic"
REPO_DIR = "/content/MangaTR"
PUBLIC_URL = "https://github.com/Nader-Geek/MangaTR.git"

if not os.path.isdir(REPO_DIR):
    try:
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = None
    url = (
        PUBLIC_URL.replace("https://", f"https://x-access-token:{token}@") if token else PUBLIC_URL
    )
    done = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, url, REPO_DIR],
        capture_output=True,
        text=True,
    )
    shown = (done.stdout + done.stderr).replace(token or "\0", "***")
    print(shown[-800:])
    # لا تخلّي التوكن مخزّن في إعدادات الـ clone
    subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", PUBLIC_URL])
else:
    print("الكود موجود بالفعل:", REPO_DIR)
print(
    subprocess.run(
        ["git", "-C", REPO_DIR, "log", "--oneline", "-1"], capture_output=True, text=True
    ).stdout
)

In [ ]:
# 3) التثبيت (onnxruntime-gpu + huggingface_hub)
%pip install -q -e "/content/MangaTR[gpu]" pytest

In [ ]:
# 4) اختبارات المشروع (من غير GPU). المتوقع: كلها passed، وممكن بعضها skipped.
!cd /content/MangaTR && python -m pytest -q

In [ ]:
# 5) الفحص: تحميل الموديلات وقياس الزمن (بياخد دقائق، التحميل أغلبه)
%cd /content/MangaTR
!python -m mangatr.gpu_check --out /content/report.json --models-dir /content/models

## ابعتلي النتيجة

- انسخ **كل اللي بين** `===== MANGATR REPORT BEGIN =====` و`===== MANGATR REPORT END =====` من خرج الخلية 5 والصقه في الشات.
- لو خلية 4 (الاختبارات) فيها failed، الصق آخر 15 سطر منها.
- لو خلية فشلت قبل التقرير، الصق رسالة الخطأ (من غير التوكن).

**الحاجات اللي هدور عليها في التقرير:**

| الحقل | معناه |
| --- | --- |
| `environment.gpu` | اسم الكارت والـ VRAM |
| `environment.available_providers` | لازم فيها `CUDAExecutionProvider` |
| `models.*.download.sha_ok` | `true` يعني الموديل هو المتوقع |
| `models.*.gpu.ran_on_gpu` | `false` يعني onnxruntime رجع للـ CPU (مشكلة تثبيت CUDA) |
| `models.*.gpu.ms.median` / `models.*.cpu.ms.median` | الزمن بالملّي ثانية على الـ GPU والـ CPU |
| `models.*.gpu.gpu_memory_used_mb` | الـ VRAM المستخدمة والموديل محمّل |
